# Filter initial dataset for plots


The `LeadVariantEffect` dataset has to be filtered for downstream analysis.

For the downstream analysis of MAF, rescaled effect size and variant effects we need to filter the `LeadVariantEffect` dataset to:

1. Limit the dataset to `gwas, cis-pqtl and eqlt` datasets.
2. Split the GWAS studies into two types:
   - measurements (continuous traits)
   - diseases (binary traits bound to therapeutic areas)
3. Apply the `replicated` mask to the GWAS-measurements, GWAS-diseases and molecular QTL datasets.
4. Apply the `qualified` mask to the GWAS-measurements, GWAS-diseases.
5. Apply Posterior Inclusion Probability (PIP) filter to all lead variants, keeping only those with PIP >= 0.5.

After the filtering applied check how many variants are left in the dataset and compute statistics on rescaled effect size & MAF.

5. Ensure there is no lead variants with MAF >= 0.01 and is not empty
6. Ensure there is no lead variants with absolute value of rescaled effect size <= 3 and is not empty

The resulting dataset should be split into two datasets:

- lead variant effects with MAF >= 0.01 (qualified_lead_variant_effect_maf_filtered)
- lead variant effects with all variants (qualified_lead_variant_effect)


## Data Loading

The data required for the analysis is loaded from the

- `lead variant effect` dataset
- `qualified gwas measurements` dataset
- `qualified gwas diseases` dataset
- `replicated molecular qtls` dataset
- `replicated gwas` dataset


### Data file paths


In [1]:
qualifying_gwas_disease_credible_set_path = "../../../data/intermediate_files/qualifying_credible_sets"
qualifying_gwas_measurements_credible_set_path = "../../../data/intermediate_files/qualifying_measurement_credible_sets"
lead_variant_effect_dataset_path = "../../../data/intermediate_files/lead_variant_effect"

replicated_molqtls_path = "../../../data/intermediate_files/list_of_molqtls_replicated_CSs.parquet"
replicated_gwas_path = "../../../data/intermediate_files/list_of_gwas_replicated_CSs.parquet"
replicated_credible_sets_path = "../../data/replicated_credible_sets"


# Output paths
qualified_lve_path = "../../../data/intermediate_files/qualified_lead_variant_effect"


### Data reading


In [ ]:
from gentropy.common.session import Session
from pyspark.sql import functions as f

from manuscript_methods import group_statistics
from manuscript_methods.datasets import LeadVariantEffect
from manuscript_methods.locus_statistics import LocusStatistics
from manuscript_methods.rescaled_beta import RescaledStatistics
from manuscript_methods.study_statistics import StudyStatistics, StudyType


Loading BokehJS ...

/Users/ss60/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [3]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
full_lve = LeadVariantEffect.from_parquet(session=session, path=lead_variant_effect_dataset_path)
qualified_gwas_measurements_lve = LeadVariantEffect.from_parquet(
    session=session, path=qualifying_gwas_measurements_credible_set_path
)
qualified_gwas_disease_lve = LeadVariantEffect.from_parquet(
    session=session, path=qualifying_gwas_disease_credible_set_path
)
replicated_molqtls = session.spark.read.parquet(replicated_molqtls_path)
replicated_gwas = session.spark.read.parquet(replicated_gwas_path)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/16 09:32:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
full_lve.df.show(1)


+---------------+--------------------+--------------------+--------------------+---------------+----------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------------+------------------------+
|      variantId|             variant|        studyLocusId|             studyId|         geneId|diseaseIds|originalBeta|originalStandardError|     locusStatistics|finemappingMethod|isTransQtl|       variantEffect|majorLdPopulation|majorLdPopulationMaf| majorLdPopulationAf|   variantStatistics|     studyStatistics|  rescaledStatistics|leadVariantConsequence|traitFromSourceMappedIds|
+---------------+--------------------+--------------------+--------------------+---------------+----------+------------+---------------------+--------------------+-----------------+----------+--------------------+-----------------

## Analysis steps

1. Filter full lve dataset to keep only `cis-pqtl` and `eqtl` signals.
2. Union with the `qualified gwas measurements` and `qualified gwas diseases` lve datasets
3. Limit the signals to the replicated ones (inner join with replicated gwas and replicated molqtls)
4. Remove lead variants with MAF == 0.0 and empty MAF 
5. Remove lead variants where absolute value of rescaled effect size <= 3
6. Filter by PIP >= 0.5
7. Save the filtered dataset to parquet
8. MAF filter >= 0.01
9. Save the filtered datasets


In [5]:
PIP_THRESHOLD = 0.1
RESCALED_BETA_ABS_THRESHOLD = 3
MAF_THRESHOLD = None


#### Filter based on cis-pqtl and eqtl signals

In [6]:
print("Filtering to molecular QTLs (cis-pQTLs and eQTLs) and removing trans-QTLs")
print(f"Before filtering qtls: {full_lve.df.count():,}")
molqtl_lve = LeadVariantEffect(
    full_lve.df.filter(
        StudyStatistics().study_type.isin(
            StudyType.CIS_PQTL,
            StudyType.EQTL,
        )
    ).filter(~f.col("isTransQtl"))
)
print(f"After filtering qtls: {molqtl_lve.df.count():,}")


Filtering to molecular QTLs (cis-pQTLs and eQTLs) and removing trans-QTLs
Before filtering qtls: 2,833,758
After filtering qtls: 1,365,531


#### Filter based on qualified gwas measurements and diseases datasets

In [7]:
print("Filtering based on qualified gwas measurements and diseases datasets")
print(f"Before union - qualified measurements: {qualified_gwas_measurements_lve.df.count():,}")
print(f"Before union - qualified disease: {qualified_gwas_disease_lve.df.count():,}")
print(f"Before union - molqtl: {molqtl_lve.df.count():,}")

study_stats = StudyStatistics()

# Adjust the study types
qualified_gwas_measurements_lve = LeadVariantEffect(
    qualified_gwas_measurements_lve.df.withColumn(
        study_stats.name, study_stats.transform_study_type(StudyType.GWAS_MEASUREMENT).col
    )
)
qualified_gwas_disease_lve = LeadVariantEffect(
    qualified_gwas_disease_lve.df.withColumn(
        study_stats.name, study_stats.transform_study_type(StudyType.GWAS_DISEASE).col
    )
)


qualified_lve = LeadVariantEffect(
    molqtl_lve.df.unionByName(qualified_gwas_measurements_lve.df).unionByName(qualified_gwas_disease_lve.df)
)
print(f"After union: {qualified_lve.df.count():,}")


Filtering based on qualified gwas measurements and diseases datasets
Before union - qualified measurements: 450,357
Before union - qualified disease: 70,618
Before union - molqtl: 1,365,531
After union: 1,886,506


#### Filter based on replicated credible sets

In [8]:
# Filter based on replicated credible sets
print(f"Before filter - qualified credible sets: {qualified_lve.df.count():,}")
print(f"Before filter - replicated GWAS measurement and disease credible sets: {replicated_gwas.count():,}")
print(f"Before filter - replicated molQTL credible sets: {replicated_molqtls.count():,}")
replicated_cs_df = replicated_gwas.unionByName(replicated_molqtls).cache()
cs_count = replicated_cs_df.count()
print(f"Before filter - replicated credible sets: {cs_count:,}")
replicated_qualified_lve = qualified_lve.filter_by_study_locus_id(replicated_cs_df)
print(f"After filter - qualified replicated credible sets: {replicated_qualified_lve.df.count():,}")


Before filter - qualified credible sets: 1,886,506
Before filter - replicated GWAS measurement and disease credible sets: 263,705
Before filter - replicated molQTL credible sets: 1,461,445


Before filter - replicated credible sets: 1,725,150


StudyLocus dimension: 1725150, unique studyLocusId: 1725150


Initial rows: 1886506
Filtered 749416 rows based on the StudyLocus.
Remaining rows: 1137090


After filter - qualified replicated credible sets: 1,137,090


#### Filter MAF and abs effect size outliers

In [11]:
print("Filtering lead variant effects based on MAF and effect size outliers")
print(f"Before filter - qualified replicated credible sets: {replicated_qualified_lve.df.count():,}")
lve_filtered = replicated_qualified_lve.maf_filter(
    remove_null=True, remove_zero=True, threshold=MAF_THRESHOLD
).effect_size_filter(effect_size_threshold=RESCALED_BETA_ABS_THRESHOLD)
print(f"After filter - qualified replicated credible sets: {lve_filtered.df.count():,}")


Filtering lead variant effects based on MAF and effect size outliers


Before filter - qualified replicated credible sets: 1,137,090


After filter - qualified replicated credible sets: 1,117,120


#### Filter based on PIP threshold

In [12]:
print("Filtering lead variant effects based on PIP outliers")
locus_stats = LocusStatistics()
print(f"Before filter - qualified replicated credible sets: {lve_filtered.df.count():,}")
lve_final = lve_filtered.filter(locus_stats.col.getField("leadVariantPIP") >= PIP_THRESHOLD)
print(f"After filter - qualified replicated credible sets: {lve_final.df.count():,}")


Filtering lead variant effects based on PIP outliers


Before filter - qualified replicated credible sets: 1,117,120


After filter - qualified replicated credible sets: 950,517


#### Save the dataset

In [13]:
lve_final.df.write.mode("overwrite").parquet(qualified_lve_path)


## Sanity check

Post filtering sanity checks to ensure the filtering steps were successful.


### Check the number of rare variants 

In [14]:
# Apply MAF filter to 0.01
print(f"Before MAF filtering: {lve_final.df.count():,}")
lve_maf_filtered = lve_final.maf_filter()
print(f"After MAF filtering: {lve_maf_filtered.df.count():,}")


Before MAF filtering: 950,517


After MAF filtering: 938,285


### Check the number of studyTypes after all filtering


In [15]:
group_statistics(lve_final.df.select("studyStatistics.studyType"), [f.col("studyType")]).show()


+----------------+------+-----+------------------+
|       studyType| count|    %|        percentage|
+----------------+------+-----+------------------+
|            eqtl|778127|81.86| 81.86355425521057|
|gwas-measurement|148627|15.64|15.636437854346635|
|    gwas-disease| 21756| 2.29|2.2888596416476505|
|        cis-pqtl|  2007| 0.21|0.2111482487951294|
+----------------+------+-----+------------------+



### Check abs beta distibution

In [16]:
data = (
    lve_final.df.select(
        f.col("rescaledStatistics.absEstimatedBeta").alias("beta"),
        f.col("studyStatistics.studyType").alias("studyType"),
    )
    .groupBy("studyType")
    .agg(
        f.avg("beta").alias("beta_mean"),
        f.stddev("beta").alias("beta_stddev"),
        f.expr("percentile_approx(beta, 0.25)").alias("beta_quartile_1"),
        f.expr("percentile_approx(beta, 0.75)").alias("beta_quartile_3"),
        f.expr("percentile_approx(beta, 0.5)").alias("beta_median"),
    )
)
data.show(truncate=False)


+----------------+-------------------+-------------------+--------------------+-------------------+-------------------+
|studyType       |beta_mean          |beta_stddev        |beta_quartile_1     |beta_quartile_3    |beta_median        |
+----------------+-------------------+-------------------+--------------------+-------------------+-------------------+
|gwas-measurement|0.07671673758048735|0.14783767648514576|0.020099253957145843|0.07184382103213831|0.0338795576423011 |
|cis-pqtl        |0.3859950876877113 |0.30274144736630487|0.18829912736982185 |0.45683380122837075|0.30423910516364666|
|gwas-disease    |0.17351417657402476|0.22991116995282046|0.05848079850730898 |0.18504458755749362|0.09700852313099388|
|eqtl            |0.92021212285408   |0.43105635921348306|0.6018472312646522  |1.1391803670571772 |0.8248332736782921 |
+----------------+-------------------+-------------------+--------------------+-------------------+-------------------+



In [ ]:
session.spark.stop()
